# 🚀 HiPerGator Batch Job (SLURM) Setup

This section creates a ready-to-run SLURM job script for executing the full notebook on HiPerGator.

- The next cell writes `submit_job.sh` (as executable code).
- Then you can submit with `sbatch submit_job.sh`.
- A usage cell below includes all common monitoring/debug prompts.

In [ ]:
'''


from pathlib import Path
import os

job_script = """#!/bin/bash
#SBATCH --job-name=medical_digital_twin
#SBATCH --account=prismap-ai-core
#SBATCH --qos=prismap-ai-core
#SBATCH --partition=hpg-b200
#SBATCH --gres=gpu:b200:1  # Request B200 GPU (confirm availability)
#SBATCH --mem=64GB
#SBATCH --time=12:00:00
#SBATCH --output=job_output_%j.log
#SBATCH --error=job_error_%j.log

set -euo pipefail

# Ensure we are in submission directory
cd "${SLURM_SUBMIT_DIR:-$(pwd)}"

echo "[$(date)] Starting job ${SLURM_JOB_ID:-N/A} on $(hostname)"

# Load and activate conda environment (robust for non-interactive shells)
module purge
module load conda

if command -v conda >/dev/null 2>&1; then
    source "$(conda info --base)/etc/profile.d/conda.sh"
else
    echo "ERROR: conda command not found after module load." >&2
    exit 1
fi

conda activate digitaltwins_env

echo "Python: $(which python)"
echo "Jupyter: $(which jupyter || true)"

# Avoid ~/.local site-packages shadowing env packages on compute nodes
export PYTHONNOUSERSITE=1

if ! command -v jupyter >/dev/null 2>&1; then
    echo "ERROR: jupyter is not available in digitaltwins_env." >&2
    exit 1
fi

# Run the notebook (assuming jupyter is available in the environment)
jupyter nbconvert \\
    --to notebook \\
    --execute medical_digital_twin_master.ipynb \\
    --ExecutePreprocessor.timeout=-1 \\
    --output executed_notebook.ipynb

echo "[$(date)] Notebook execution completed. Output: executed_notebook.ipynb"

# Usage note: open in VS Code from login node after job finishes
echo "To open in VS Code from login node: code executed_notebook.ipynb"
"""

script_path = Path("submit_job.sh")
script_path.write_text(job_script)
os.chmod(script_path, 0o755)

print(f"✅ Wrote executable job script: {script_path.resolve()}")

'''

✅ Wrote executable job script: /blue/prismap-ai-core/Ahmed/DigitalTwins/MDT/submit_job.sh


In [ ]:
'''
usage_prompts = [
    "sbatch submit_job.sh",
    "squeue -u $USER",
    "sacct -j <JOB_ID> --format=JobID,JobName,Partition,State,Elapsed,MaxRSS",
    "tail -f job_output_<JOB_ID>.log",
    "tail -f job_error_<JOB_ID>.log",
    "ls -lh executed_notebook.ipynb",
    "code executed_notebook.ipynb"
]

print("📌 HiPerGator usage prompts:")
for i, cmd in enumerate(usage_prompts, start=1):
    print(f"{i}. {cmd}")

print("\nTip: replace <JOB_ID> with the value returned by sbatch.")
'''

📌 HiPerGator usage prompts:
1. sbatch submit_job.sh
2. squeue -u $USER
3. sacct -j <JOB_ID> --format=JobID,JobName,Partition,State,Elapsed,MaxRSS
4. tail -f job_output_<JOB_ID>.log
5. tail -f job_error_<JOB_ID>.log
6. ls -lh executed_notebook.ipynb
7. code executed_notebook.ipynb

Tip: replace <JOB_ID> with the value returned by sbatch.


In [ ]:
#!sbatch submit_job.sh

Submitted batch job 28308162


In [ ]:
#!squeue -u $USER

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          28308177  hpg-b200 medical_ ahmed.so PD       0:00      1 (Priority)
          28271999  hpg-b200 ondemand ahmed.so  R   10:17:58      1 c1009a-s15


In [ ]:
#!sacct -j 28307574 --format=JobID,JobName,Partition,State,Elapsed,MaxRSS

JobID           JobName  Partition      State    Elapsed     MaxRSS 
------------ ---------- ---------- ---------- ---------- ---------- 
28307574     medical_d+   hpg-b200     FAILED   00:00:11            
28307574.ba+      batch                FAILED   00:00:11      2696K 
28307574.ex+     extern             COMPLETED   00:00:11            


# Metacognitive Medical Digital Twins Pipeline

This notebook implements the complete MDT pipeline with MIMIC-IV and Medical-O1 data sources.

**Alignment Components:**
- Theory of Mind module for user belief inference
- Composite reward engine (5 components: safety, empathy, proactivity, metacognition, semantic)
- GRPO training for multi-objective alignment

**Ontology Components:**
- LOINC, SNOMED-CT, ICD-10 code mappings
- Clinical reference ranges validation
- MIMIC-IV item ID mappings

**Data Sources:**
- MIMIC-IV (ICU trajectories)
- Medical-O1 (reasoning chains)


## 1. Setup & Environment

In [1]:
# Environment setup
import sys
sys.path.append(".")

# Core imports
from config.configs import DataConfig
from data.mimic_processor import MIMICProcessor
from data.medical_o1_processor import MedicalO1Processor
from core.theory_of_mind import TheoryOfMindModule
from rewards.composite_engine import CompositeRewardEngine
from training.grpo_trainer import run_grpo_training
from utils.ontology_validator import OntologyValidator
from utils.helpers import clean_memory, setup_logging

print("✓ All components imported successfully")


✓ All components imported successfully


## 2. Configuration

In [ ]:
# Setup logging
setup_logging()
import logging
logger = logging.getLogger(__name__)

logger.info("="*80)
logger.info("MEDICAL DIGITAL TWIN - MASTER PIPELINE")
logger.info("="*80)

# Load configuration
data_config = DataConfig()
print(f"✓ Configuration loaded")
print(f"  MIMIC patients: {data_config.max_patients}")
print(f"  Medical-O1 examples: {data_config.max_o1_examples}")


NameError: name 'setup_logging' is not defined

: 

: 

: 

## 3. Data Loading

In [ ]:
# Load MIMIC-IV data
print("Loading MIMIC-IV data...")
mimic_processor = MIMICProcessor(data_config)

if mimic_processor.check_availability():
    mimic_data = mimic_processor.process_all_patients(
        max_patients=data_config.max_patients
    )
    print(f"✓ Loaded {len(mimic_data)} MIMIC examples")
    
    # Load ICD diagnoses for validation
    try:
        icd_diagnoses = mimic_processor.load_icd_diagnoses()
        print(f"✓ Loaded {len(icd_diagnoses)} ICD diagnoses")
    except Exception as e:
        print(f"⚠️ ICD diagnoses loading failed: {e}")
        icd_diagnoses = None
else:
    print("⚠️ MIMIC data not available")
    mimic_data = []
    icd_diagnoses = None

Loading MIMIC-IV data...
2026-03-26 18:08:26,412 - data.mimic_processor - INFO - Initialized MIMIC-IV v3.1 processor
2026-03-26 18:08:26,413 - data.mimic_processor - INFO - Root directory: mimiciv/3.1
2026-03-26 18:08:26,413 - data.mimic_processor - INFO - Loaded 61 LOINC codes from ontology
2026-03-26 18:08:26,413 - data.mimic_processor - INFO - Loaded 19 chartevents item categories
2026-03-26 18:08:26,414 - data.mimic_processor - INFO - Loaded 32 reference ranges
2026-03-26 18:08:26,415 - data.mimic_processor - INFO - All required MIMIC-IV files found
2026-03-26 18:08:26,415 - data.mimic_processor - INFO - Processing up to 1000 patients...
2026-03-26 18:08:26,415 - data.mimic_processor - INFO - Loading ICU stays from: mimiciv/3.1/icu/icustays.csv
2026-03-26 18:08:26,568 - data.mimic_processor - INFO - Loaded 94458 ICU stays
2026-03-26 18:08:26,576 - data.mimic_processor - INFO - Selected 1000 ICU stays
2026-03-26 18:08:26,577 - data.mimic_processor - INFO - Extracting vital signs for

: 

: 

: 

In [ ]:
# Load Medical-O1 data
print("Loading Medical-O1 data...")
o1_processor = MedicalO1Processor()

try:
    o1_dataset = o1_processor.load_dataset(
        split='train',
        config=data_config.medical_o1_config
    )
    if o1_dataset:
        o1_data = o1_processor.format_for_training(
            o1_dataset,
            max_examples=data_config.max_o1_examples
        )
        print(f"✓ Loaded {len(o1_data)} Medical-O1 examples")
    else:
        print("⚠️ Medical-O1 dataset loading failed")
        o1_data = []
except Exception as e:
    print(f"⚠️ Medical-O1 loading failed: {e}")
    o1_data = []

# Combine datasets
all_training_data = mimic_data + o1_data
print(f"✓ Total training examples: {len(all_training_data)}")
print(f"  MIMIC-IV: {len(mimic_data)}")
print(f"  Medical-O1: {len(o1_data)}")

Loading Medical-O1 data...
2026-03-26 18:08:30,405 - data.medical_o1_processor - INFO - Initialized MedicalO1Processor
2026-03-26 18:08:30,405 - data.medical_o1_processor - INFO - Loading FreedomIntelligence/medical-o1-reasoning-SFT (config=en) from HuggingFace...
2026-03-26 18:08:30,516 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-26 18:08:30,528 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/fc2c9e8a37b38f38da6d449564a8c350b244aef4/README.md "HTTP/1.1 200 OK"
2026-03-26 18:08:30,573 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/resolve/fc2c9e8a37b38f38da6d449564a8c350b244aef4/medical-o1-reasoning-SFT.py "HTTP/1.1 404 Not Found"
2026-03-26 18:08:30,676 - httpx - INFO - HTTP Request: HEAD https://s3.amazona

2026-03-26 18:08:31,018 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-26 18:08:31,024 - data.medical_o1_processor - INFO - ✓ Loaded 19704 examples from Medical-O1 dataset
2026-03-26 18:08:31,025 - data.medical_o1_processor - INFO - Formatting 5000 Medical-O1 examples...
2026-03-26 18:08:31,382 - data.medical_o1_processor - INFO - ✓ Formatted 5000 Medical-O1 examples
✓ Loaded 5000 Medical-O1 examples
✓ Total training examples: 5160
  MIMIC-IV: 160
  Medical-O1: 5000


: 

: 

: 

## 3.5. Ontology Validation

In [ ]:
# Initialize ontology validator
print("Initializing ontology validator...")
ontology_validator = OntologyValidator()

# Validate ICD codes if available
if icd_diagnoses is not None:
    print("Validating ICD-10 codes...")
    valid_icd_count = 0
    total_icd_count = len(icd_diagnoses)
    
    # Sample validation (check first 100 for speed)
    sample_icd = icd_diagnoses.head(100)
    for _, row in sample_icd.iterrows():
        icd_code = row['icd_code']
        if ontology_validator.validate_icd10_code(icd_code):
            valid_icd_count += 1
    
    print(f"✓ ICD-10 validation: {valid_icd_count}/{len(sample_icd)} sample codes valid")
    print(f"  Total ICD diagnoses available: {total_icd_count}")
else:
    print("⚠️ ICD diagnoses not available for validation")

# Validate LOINC codes from loaded data
print("Validating LOINC codes...")
loinc_codes = ontology_validator.loinc_codes
print(f"✓ Loaded {len(loinc_codes)} LOINC codes in ontology")

# Test some common LOINC codes
test_loinc = ['8867-4', '8480-6', '8462-4', '2524-7', '2160-0']  # HR, SBP, DBP, Lactate, Creatinine
valid_loinc_count = 0
for code in test_loinc:
    if ontology_validator.validate_loinc_code(code):
        valid_loinc_count += 1

print(f"✓ LOINC validation: {valid_loinc_count}/{len(test_loinc)} test codes valid")

# Validate SNOMED-CT codes
print("Validating SNOMED-CT codes...")
snomed_codes = ontology_validator.snomed_codes
print(f"✓ Loaded {len(snomed_codes)} SNOMED-CT codes in ontology")

# Test some common SNOMED codes
test_snomed = ['91302008', '233604007', '67782005', '84114007', '22298006']  # Sepsis, Pneumonia, ARDS, Heart Failure, MI
valid_snomed_count = 0
for code in test_snomed:
    if ontology_validator.validate_snomed_code(code):
        valid_snomed_count += 1

print(f"✓ SNOMED-CT validation: {valid_snomed_count}/{len(test_snomed)} test codes valid")

print("✓ Complete ontology validation completed")

Initializing ontology validator...
2026-03-26 18:08:31,389 - utils.ontology_validator - INFO - Initialized OntologyValidator
2026-03-26 18:08:31,390 - utils.ontology_validator - INFO -   LOINC codes: 61
2026-03-26 18:08:31,390 - utils.ontology_validator - INFO -   SNOMED codes: 50
2026-03-26 18:08:31,391 - utils.ontology_validator - INFO -   ICD-10 codes: 52
2026-03-26 18:08:31,391 - utils.ontology_validator - INFO -   Reference ranges: 32
Validating ICD-10 codes...
✓ ICD-10 validation: 2/100 sample codes valid
  Total ICD diagnoses available: 6364488
Validating LOINC codes...
✓ Loaded 61 LOINC codes in ontology
✓ LOINC validation: 5/5 test codes valid
Validating SNOMED-CT codes...
✓ Loaded 50 SNOMED-CT codes in ontology
✓ SNOMED-CT validation: 5/5 test codes valid
✓ Complete ontology validation completed


: 

: 

: 

## 4. Model Training (SFT)

In [ ]:
# Import SFT training components
from training.sft_trainer import run_sft_training
from models.mdt_model import MedicalDigitalTwinModel
from config.configs import ModelConfig, SFTConfig
from data.dataset import CognitiveStreamDataset
import os
import json

# Initialize model configuration
model_config = ModelConfig()
print(f"✓ Model config loaded: {model_config.model_name}")

# Initialize SFT training configuration
sft_config = SFTConfig()
print(f"✓ SFT config loaded: {sft_config.num_epochs} epochs")

# Initialize model
model = MedicalDigitalTwinModel(model_config)

# Prepare training data split
if len(all_training_data) > 0:
    # Split data for training and evaluation (90% train, 10% eval)
    split_idx = int(len(all_training_data) * 0.9)
    train_data = all_training_data[:split_idx]
    eval_data = all_training_data[split_idx:]
    
    # Create dataset objects
    train_dataset = CognitiveStreamDataset(
        train_data,
        model.tokenizer,
        max_length=model_config.max_length
    )
    
    eval_dataset = CognitiveStreamDataset(
        eval_data,
        model.tokenizer,
        max_length=model_config.max_length
    )
    
    print(f"✓ Data split: {len(train_dataset)} train, {len(eval_dataset)} eval examples")
else:
    train_dataset = None
    eval_dataset = None
    print("⚠️ No training data available")

# Run SFT training
print("Starting SFT training...")
if train_dataset is not None and eval_dataset is not None:
    trained_model = run_sft_training(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        config=sft_config
    )
    print("✓ SFT training completed")

    # Post-run diagnostics for under-training / no-step runs
    history_path = "outputs/sft/training_history.json"
    if os.path.exists(history_path):
        try:
            with open(history_path, "r") as f:
                history = json.load(f)

            total_steps = int(history.get("total_steps", 0) or 0)
            losses = history.get("training_losses", [])

            if total_steps == 0:
                print("⚠️ Diagnostic: total_steps=0. Optimizer updates did not occur.")
                print("   Likely cause: too few batches vs gradient_accumulation_steps.")
                print("   Consider reducing SFTConfig.gradient_accumulation_steps or increasing training data.")

            if len(losses) <= 1:
                print(f"⚠️ Diagnostic: only {len(losses)} training loss point(s) logged.")
                print("   The loss curve is not yet meaningful; run with more steps/data.")
        except Exception as e:
            print(f"⚠️ Could not parse SFT history diagnostics: {e}")
    else:
        print("⚠️ SFT history file not found; unable to run post-training diagnostics.")
else:
    print("⚠️ Skipping SFT training - no data available")
    trained_model = model

✓ Model config loaded: Qwen/Qwen3.5-4B
✓ SFT config loaded: 3 epochs
2026-03-26 18:09:05,736 - models.mdt_model - WARNING - Using GPT-2 for demo. Set use_demo_model=False for production.
2026-03-26 18:09:05,786 - models.mdt_model - INFO - Loading GPT-2 demo model...
2026-03-26 18:09:05,852 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-26 18:09:05,896 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-26 18:09:05,942 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

2026-03-26 18:09:06,152 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/generation_config.json "HTTP/1.1 200 OK"
2026-03-26 18:09:06,203 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-26 18:09:06,246 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-26 18:09:06,292 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-26 18:09:06,336 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-26 18:09:06,381 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main?recursive=true&expan

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


2026-03-26 18:09:07,656 - models.mdt_model - INFO - Demo model loaded successfully
2026-03-26 18:09:07,657 - data.dataset - INFO - Initialized dataset with 4644 examples
2026-03-26 18:09:07,657 - data.dataset - INFO - Initialized dataset with 516 examples
✓ Data split: 4644 train, 516 eval examples
Starting SFT training...
2026-03-26 18:09:07,658 - training.sft_trainer - INFO - ================================================================================
2026-03-26 18:09:07,658 - training.sft_trainer - INFO - STARTING SUPERVISED FINE-TUNING (PHASE 1)
2026-03-26 18:09:07,659 - training.sft_trainer - INFO - ================================================================================
2026-03-26 18:09:07,659 - training.sft_trainer - INFO - 
2026-03-26 18:09:07,659 - training.sft_trainer - INFO - Training Configuration:
2026-03-26 18:09:07,659 - training.sft_trainer - INFO -   Model: Qwen/Qwen3.5-4B
2026-03-26 18:09:07,660 - training.sft_trainer - INFO -   Training examples: 4644
202

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-26 18:11:14,126 - training.sft_trainer - INFO - 
2026-03-26 18:11:14,126 - training.sft_trainer - INFO - ================================================================================
2026-03-26 18:11:14,127 - training.sft_trainer - INFO - SUPERVISED FINE-TUNING COMPLETE
2026-03-26 18:11:14,127 - training.sft_trainer - INFO - ================================================================================
2026-03-26 18:11:14,127 - training.sft_trainer - INFO -   Final loss: 1.2716
2026-03-26 18:11:14,127 - training.sft_trainer - INFO -   Total steps: 438
2026-03-26 18:11:14,127 - training.sft_trainer - INFO -   Model saved to: outputs/sft/final_model
2026-03-26 18:11:14,128 - training.sft_trainer - INFO -   Logs saved to: outputs/logs/sft
2026-03-26 18:11:14,128 - training.sft_trainer - INFO - ================================================================================
2026-03-26 18:11:14,128 - training.sft_trainer - INFO - 
✓ SFT training completed


: 

: 

: 

In [3]:
# View SFT Training Results
import json
import os
import pandas as pd
import matplotlib.pyplot as plt

print("--- SFT Training Results ---")

def _fmt_float(value, digits=4):
    try:
        return f"{float(value):.{digits}f}"
    except (TypeError, ValueError):
        return "N/A"

metrics_path = "outputs/sft/training_metrics.json"
has_trainer_metrics = os.path.exists(metrics_path)

if has_trainer_metrics:
    with open(metrics_path, "r") as f:
        metrics = json.load(f)
    
    print(f"Final Training Loss: {_fmt_float(metrics.get('train_loss'))}")
    print(f"Total Steps: {metrics.get('total_steps', 'N/A')}")
    print(f"Total Epochs: {metrics.get('epoch', 'N/A')}")
    print(f"Runtime: {_fmt_float(metrics.get('train_runtime'), 2)} seconds")
    print(f"Samples per second: {metrics.get('train_samples_per_second', 'N/A')}")
    print("✓ Using Hugging Face Trainer metrics (authoritative for this run).")
else:
    print("Could not find Hugging Face Trainer metrics. Checking for basic loop history...")

    history_path = "outputs/sft/training_history.json"
    if os.path.exists(history_path):
        with open(history_path, "r") as f:
            history = json.load(f)

        losses = history.get('training_losses', [])
        if losses:
            epochs = list(range(1, len(losses) + 1))

            plt.figure(figsize=(8, 4))
            plt.plot(epochs, losses, marker='o', label='Training Loss')
            plt.title('SFT Training Curve')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.xticks(epochs)
            plt.grid(True)
            plt.legend()

            if len(losses) == 1:
                plt.xlim(0.5, 1.5)
                print("⚠️ Only one loss point found (single epoch/summary). Curve shape is not meaningful yet.")

            plt.show()

            print(f"History points: {len(losses)}")
            print(f"First loss: {losses[0]:.4f} | Last loss: {losses[-1]:.4f}")
        else:
            print("⚠️ training_history.json exists but contains no 'training_losses'.")
    else:
        print("⚠️ No SFT history file found at outputs/sft/training_history.json")

# If tensorboard logs exist, we can enable it inline
print("\nTo view detailed interactive loss curves, you can run this command in a new cell:")
print("%load_ext tensorboard")
print("%tensorboard --logdir outputs/sft/runs/")

--- SFT Training Results ---
Final Training Loss: 1.2715
Total Steps: 438
Total Epochs: 3.0
Runtime: 126.14 seconds
Samples per second: 110.449
✓ Using Hugging Face Trainer metrics (authoritative for this run).

To view detailed interactive loss curves, you can run this command in a new cell:
%load_ext tensorboard
%tensorboard --logdir outputs/sft/runs/


In [4]:
%load_ext tensorboard
%tensorboard --logdir outputs/sft/runs/

## 5. Alignment Training (GRPO)

In [ ]:
# Import GRPO training components
from pathlib import Path
import json
from training.grpo_trainer import run_grpo_training
from config.configs import GRPOConfig
from data.dataset import CognitiveStreamDataset
from torch.utils.data import DataLoader

# Initialize reward engine
reward_engine = CompositeRewardEngine()

# Create GRPO configuration
grpo_config = GRPOConfig()
print(f"✓ GRPO config loaded: {grpo_config.num_iterations} default iterations")

# --- Resume-aware GRPO setup ---
grpo_output_dir = Path(grpo_config.output_dir)
grpo_output_dir.mkdir(parents=True, exist_ok=True)
resume_chunk = 200  # Increase target by this many iterations when resuming

def latest_completed_iteration(grpo_path: Path) -> int:
    latest = 0
    if not grpo_path.exists():
        return latest

    # From iteration_* checkpoint folders
    for child in grpo_path.iterdir():
        if child.is_dir() and child.name.startswith("iteration_"):
            try:
                latest = max(latest, int(child.name.replace("iteration_", "", 1)))
            except ValueError:
                pass

    # From training_stats.json (if present)
    stats_path = grpo_path / "training_stats.json"
    if stats_path.exists():
        try:
            with open(stats_path, "r") as f:
                stats = json.load(f)
            if isinstance(stats, dict) and stats.get("iterations"):
                latest = max(latest, int(stats["iterations"][-1]))
        except Exception as e:
            print(f"⚠️ Could not parse training_stats.json for resume info: {e}")

    return latest

latest_iter = latest_completed_iteration(grpo_output_dir)
if latest_iter > 0:
    # Ensure new run continues beyond latest checkpoint
    grpo_config.num_iterations = max(grpo_config.num_iterations, latest_iter + resume_chunk)
    print(f"✓ Resume checkpoint detected at iteration {latest_iter}")
    print(f"✓ New GRPO target iteration set to {grpo_config.num_iterations}")
else:
    print("✓ No previous GRPO checkpoint found; starting fresh GRPO run")
    print(f"✓ GRPO target iteration: {grpo_config.num_iterations}")

# Create dataloader for GRPO training
if train_dataset is not None:
    # Use batch_size=1 for GRPO since it processes prompts individually
    grpo_dataloader = DataLoader(
        train_dataset,
        batch_size=1,  # Process one prompt at a time for GRPO
        shuffle=True
    )
    print(f"✓ Created GRPO dataloader with {len(train_dataset)} examples")
else:
    grpo_dataloader = None
    print("⚠️ No training data available for GRPO")

# Run GRPO alignment (trainer auto-resumes from latest checkpoint/stats)
print("Starting GRPO alignment training...")
if grpo_dataloader is not None:
    aligned_model = run_grpo_training(
        model=trained_model,
        train_dataloader=grpo_dataloader,
        config=grpo_config,
        reward_engine=reward_engine
    )
    print("✓ GRPO alignment completed")
else:
    print("⚠️ Skipping GRPO training - no dataloader available")
    aligned_model = trained_model

2026-03-26 18:11:14,330 - absl - INFO - Using default tokenizer.
2026-03-26 18:11:14,413 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-26 18:11:14,459 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-26 18:11:14,503 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-26 18:11:14,549 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-26 18:11:14,593 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-26 18:11:14,647 - httpx - INFO - HTTP Reques

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-03-26 18:11:15,386 - rewards.semantic_reward - INFO - Initialized SemanticFidelityReward
2026-03-26 18:11:15,390 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2
2026-03-26 18:11:15,435 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-26 18:11:15,450 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json "HTTP/1.1 200 OK"
2026-03-26 18:11:15,494 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-26 18:11:15,507 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

2026-03-26 19:13:21,718 - training.grpo_trainer - INFO -   Avg Reward: 0.4932 (min: 0.3053, max: 0.5190)
2026-03-26 19:13:21,719 - training.grpo_trainer - INFO -   Policy Loss: 0.0000
2026-03-26 19:13:21,719 - training.grpo_trainer - INFO -   KL Divergence: 0.0000
2026-03-26 19:13:21,720 - training.grpo_trainer - INFO - 
2026-03-26 19:13:21,720 - training.grpo_trainer - INFO - Iteration 22/1000
2026-03-26 19:13:21,734 - training.grpo_trainer - INFO -   Generating 32 responses per prompt...


Prompts:   0%|          | 0/1 [00:00<?, ?it/s]

: 

: 

: 

In [ ]:
# Import evaluation
from evaluation.evaluator import MedicalTwinEvaluator
from core.cognitive_streams import CognitiveStreamParser

# Check if reward_engine is initialized earlier in the notebook, if not create one
if 'reward_engine' not in locals():
    print("Warning: reward_engine not found, initializing a new one for evaluation.")
    from rewards.composite_engine import CompositeRewardEngine
    reward_engine = CompositeRewardEngine()

# Initialize parser
parser = CognitiveStreamParser()

print("Initializing comprehensive evaluation framework...")
# Run evaluation
evaluator = MedicalTwinEvaluator(
    model=aligned_model,
    parser=parser,
    reward_engine=reward_engine
)

# Run the 8 predefined benchmark clinical cases
results_df = evaluator.run_evaluation()

# Generate visual report
report = evaluator.generate_report(results_df)

print("✓ Evaluation completed")
print(report)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TypeError: CognitiveStreamParser.__init__() missing 1 required positional argument: 'config'

: 

: 

: 

In [ ]:
# Import evaluation
from evaluation.evaluator import MedicalTwinEvaluator

# Run evaluation
evaluator = MedicalTwinEvaluator()
results = evaluator.evaluate_model(aligned_model)

print("✓ Evaluation completed")
print(f"Results: {results}")


TypeError: MedicalTwinEvaluator.__init__() missing 3 required positional arguments: 'model', 'parser', and 'reward_engine'

: 

: 

: 

## 7. Saving Results & Interactive Testing (Gradio)
Save the final aligned model weights, dump the evaluation metrics, and launch a simple Gradio web interface to manually test the Medical Digital Twin with hypothetical patient cases.

In [ ]:
import json
import os
import torch
from datetime import datetime

# Require GPU for Gradio/model inference
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required to run Gradio for this notebook workflow. Please run on a GPU node.")

print(f"Using GPU for Gradio: {torch.cuda.get_device_name(0)}")

# Optional: Ensure gradio is installed
try:
    import gradio as gr
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio"])
    import gradio as gr

# 1. Save Training & Evaluation Results
print("Saving results and model...")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs("results", exist_ok=True)
results_path = f"results/final_eval_results_{timestamp}.json"
report_path = f"results/evaluation_report_{timestamp}.txt"

# Save the detailed pandas dataframe as a CSV
if 'results_df' in locals():
    csv_path = f"results/evaluation_results_{timestamp}.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"✓ Full evaluation dataframe saved to {csv_path}")

# Save the formatted text report
if 'report' in locals():
    with open(report_path, "w") as f:
        f.write(report)
    print(f"✓ Summary report saved to {report_path}")

# 2. Save Trained Model
model_save_path = f"outputs/final_model_{timestamp}"
os.makedirs(model_save_path, exist_ok=True)
try:
    if hasattr(aligned_model, 'model') and hasattr(aligned_model, 'tokenizer'):
        aligned_model.model.save_pretrained(model_save_path)
        aligned_model.tokenizer.save_pretrained(model_save_path)
        print(f"✓ Model successfully saved to {model_save_path}")
    elif hasattr(aligned_model, 'save_pretrained'): # If it's a raw transformers model
        aligned_model.save_pretrained(model_save_path)
        print(f"✓ Model successfully saved to {model_save_path}")
except Exception as e:
    print(f"⚠️ Could not save model automatically: {e}")

# 3. Gradio Interface for Interactive Testing
def predict_medical_case(patient_history, current_symptoms):
    if not hasattr(aligned_model, 'model') or not hasattr(aligned_model, 'tokenizer'):
        return "Model not properly loaded! Please ensure training finished."
        
    prompt = f"Patient History: {patient_history}\n\nCurrent Symptoms: {current_symptoms}\n\nProvide a clinical assessment and recommendations:"
    
    try:
        # Generate the response using our custom evaluation generate function if possible
        if hasattr(aligned_model, 'generate'):
            raw_response = aligned_model.generate(prompt, max_length=1024)
            # Check if parser is available to extract structured streams
            if 'parser' in locals():
                streams = parser.parse(raw_response)
                formatted_response = (
                    f"🤔 **Theory of Mind (Internal)**:\n{streams.think or 'N/A'}\n\n"
                    f"📋 **Patient State**:\n{streams.patient_state or 'N/A'}\n\n"
                    f"🩺 **Final Physician Response**:\n{streams.response or raw_response}"
                )
                return formatted_response
            return raw_response
        else:
            return "aligned_model missing generate method."
            
    except Exception as e:
        return f"Error during generation: {str(e)}"

print("\n🚀 Launching Gradio Interface...")
demo = gr.Interface(
    fn=predict_medical_case,
    inputs=[
        gr.Textbox(lines=4, label="Patient History", placeholder="e.g., 45yo male with history of hypertension..."),
        gr.Textbox(lines=3, label="Current Symptoms", placeholder="e.g., severe chest pain radiating to left arm...")
    ],
    outputs=gr.Markdown(label="MDT Assessment & Recommendations"),
    title="🩺 Metacognitive Medical Digital Twin",
    description="Interactive testing interface for the trained MDT model. Simulates reasoning ('Theory of Mind') before returning an empathetic, safe physician response."
)

# Launch Gradio inline directly below
demo.launch(share=True, inline=True)

: 

: 

: 

## ⏱️ HiPerGator Walltime-Safe GRPO Resume Workflow

Use this section when SLURM walltime ends before GRPO completes.

It does three things:
1. Detects the latest `outputs/grpo/iteration_*` checkpoint.
2. Builds a resume command with a higher total target (`--grpo-iterations`).
3. Optionally regenerates training reports after each chunk.

> Recommended pattern: run GRPO in chunks (e.g., +100 or +200 iterations per job).

In [2]:
from pathlib import Path

# --- Configure chunking strategy ---
project_root = Path("/blue/prismap-ai-core/Ahmed/DigitalTwins/MDT")
grpo_dir = project_root / "outputs" / "grpo"
chunk_size = 200  # increase by this many iterations each SLURM job


def latest_completed_iteration(grpo_path: Path) -> int:
    latest = 0
    if not grpo_path.exists():
        return latest
    for child in grpo_path.iterdir():
        if child.is_dir() and child.name.startswith("iteration_"):
            try:
                latest = max(latest, int(child.name.replace("iteration_", "", 1)))
            except ValueError:
                pass
    return latest


latest_iter = latest_completed_iteration(grpo_dir)
next_target = latest_iter + chunk_size if latest_iter > 0 else chunk_size

print(f"Latest completed GRPO iteration: {latest_iter}")
print(f"Recommended next target iteration: {next_target}")

if latest_iter > 0:
    grpo_cmd = f"python main.py --train-grpo --resume-grpo --grpo-iterations {next_target}"
else:
    grpo_cmd = f"python main.py --train-grpo --grpo-iterations {next_target}"

print("\nRun this in your HiPerGator job script:")
print(grpo_cmd)

report_cmd = "python scripts/generate_training_report.py --outputs-dir outputs --report-dir results/training_reports"
print("\nThen regenerate metrics/figures:")
print(report_cmd)

Latest completed GRPO iteration: 200
Recommended next target iteration: 400

Run this in your HiPerGator job script:
python main.py --train-grpo --resume-grpo --grpo-iterations 400

Then regenerate metrics/figures:
python scripts/generate_training_report.py --outputs-dir outputs --report-dir results/training_reports


In [3]:
import subprocess

# Set to True to only print commands, False to execute report generation now
DRY_RUN = True

cmd = "python scripts/generate_training_report.py --outputs-dir outputs --report-dir results/training_reports"

if DRY_RUN:
    print("DRY_RUN=True, command not executed:")
    print(cmd)
else:
    result = subprocess.run(cmd, shell=True, cwd="/blue/prismap-ai-core/Ahmed/DigitalTwins/MDT", capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed with code {result.returncode}")

DRY_RUN=True, command not executed:
python scripts/generate_training_report.py --outputs-dir outputs --report-dir results/training_reports
